# Module 4 Homework: KNN Classification, Factor Analysis & Conjoint Analysis

This homework covers the concepts from **Module 4.0 (k-NN)** and **Module 4.1 (Factor Analysis & Conjoint Analysis)**.

**Instructions:**
- Write your code in the provided empty code cells
- Use the **exact variable names** specified in each question
- Your outputs should match the descriptions provided

---

# Part 1: k-NN Classification — Loan Default Prediction

**Dataset:** `Loan_default.csv`

A bank wants to predict which borrowers will **default** on their loans. You will build a k-NN classifier using borrower characteristics.

**Target variable:** `Default` (0 = no default, 1 = default)

### Q1: Load Libraries

Load the following libraries:
- `caret`
- `ggplot2`

In [9]:
library(caret)
library(ggplot2)

### Q2: Load and Explore the Data

Load the loan default dataset and prepare it for analysis.

- Store the data in **`loan.df`**
- Remove the `LoanID` column (not a predictor)
- Convert `Default` to a factor with labels `"No"` and `"Yes"`

**Expected output:** `str()` should show numeric columns (Age, Income, LoanAmount, CreditScore, MonthsEmployed, NumCreditLines, InterestRate, LoanTerm, DTIRatio), character/factor columns (Education, EmploymentType, MaritalStatus, HasMortgage, HasDependents, LoanPurpose, HasCoSigner), and Default as a Factor with 2 levels.

In [10]:
loan.df <- read.csv("/workspaces/MS3313_V2/data/module_4/Loan_default.csv")
loan.df$LoanID <- NULL
loan.df$Default <- factor(loan.df$Default,  levels = c(0, 1), labels = c("No", "Yes"))
str(loan.df)

'data.frame':	255347 obs. of  17 variables:
 $ Age           : int  56 69 46 32 60 25 38 56 36 40 ...
 $ Income        : int  85994 50432 84208 31713 20437 90298 111188 126802 42053 132784 ...
 $ LoanAmount    : int  50587 124440 129188 44799 9139 90448 177025 155511 92357 228510 ...
 $ CreditScore   : int  520 458 451 743 633 720 429 531 827 480 ...
 $ MonthsEmployed: int  80 15 26 0 8 18 80 67 83 114 ...
 $ NumCreditLines: int  4 1 3 3 4 2 1 4 1 4 ...
 $ InterestRate  : num  15.23 4.81 21.17 7.07 6.51 ...
 $ LoanTerm      : int  36 60 24 24 48 24 12 60 48 48 ...
 $ DTIRatio      : num  0.44 0.68 0.31 0.23 0.73 0.1 0.16 0.43 0.2 0.33 ...
 $ Education     : chr  "Bachelor's" "Master's" "Master's" "High School" ...
 $ EmploymentType: chr  "Full-time" "Full-time" "Unemployed" "Full-time" ...
 $ MaritalStatus : chr  "Divorced" "Married" "Divorced" "Married" ...
 $ HasMortgage   : chr  "Yes" "No" "Yes" "No" ...
 $ HasDependents : chr  "Yes" "No" "Yes" "No" ...
 $ LoanPurpose   : chr  "Othe

### Q3: Select Numeric Features

Create a data frame called **`knn.df`** containing only these columns from `loan.df`:

- `Age`, `Income`, `LoanAmount`, `CreditScore`, `MonthsEmployed`, `InterestRate`, `DTIRatio`, `Default`

**Expected output:** A data frame with 8 columns — 7 numeric predictors and the Default factor.

In [11]:
knn.df <- loan.df[, c("Age", "Income","LoanAmount", "CreditScore", "MonthsEmployed", "InterestRate", "DTIRatio", "Default")]
str(knn.df)

'data.frame':	255347 obs. of  8 variables:
 $ Age           : int  56 69 46 32 60 25 38 56 36 40 ...
 $ Income        : int  85994 50432 84208 31713 20437 90298 111188 126802 42053 132784 ...
 $ LoanAmount    : int  50587 124440 129188 44799 9139 90448 177025 155511 92357 228510 ...
 $ CreditScore   : int  520 458 451 743 633 720 429 531 827 480 ...
 $ MonthsEmployed: int  80 15 26 0 8 18 80 67 83 114 ...
 $ InterestRate  : num  15.23 4.81 21.17 7.07 6.51 ...
 $ DTIRatio      : num  0.44 0.68 0.31 0.23 0.73 0.1 0.16 0.43 0.2 0.33 ...
 $ Default       : Factor w/ 2 levels "No","Yes": 1 1 2 1 1 2 1 1 2 1 ...


### Q4: Split Data into Training and Holdout Sets

Split `knn.df` into training and holdout sets.

- Use `set.seed(42)`
- **60/40** split
- Store the training set in **`train.df`** and the holdout set in **`holdout.df`**

**Expected output:** Print the number of observations in each set.

In [12]:
set.seed(42)
idx <- sample(nrow(knn.df), 0.6*nrow(knn.df))
train.df <- knn.df[idx, ]
holdout.df <- knn.df[-idx, ]

nrow(train.df)
nrow(holdout.df)

[1] 153208

[1] 102139

### Q5: Train a k-NN Model with k=5

Train a k-NN model predicting `Default` from all other columns in `train.df`.

- Use **k = 5**
- **Normalize** the features (center and scale)
- Store the result in **`model`**

**Expected output:** Model summary showing k-Nearest Neighbors, the number of samples, predictors, classes, and that pre-processing was applied.

In [13]:
library(caret)
model <- train(Default ~ ., data=train.df,
method="knn",
preProcess=c("center", "scale"), 
tuneGrid=expand.grid(k=5),
trControl=trainControl(method="none"))
model

k-Nearest Neighbors 

153208 samples
     7 predictor
     2 classes: 'No', 'Yes' 

Pre-processing: centered (7), scaled (7) 
Resampling: None 

### Q6: Make Predictions on the Holdout Set

Classify the holdout set using your trained model. Store the predictions in **`predictions`**.

**Expected output:** Print the first 10 predictions — a vector of factor levels (`No` / `Yes`).

In [14]:
predictions <- predict(model, holdout.df)
head(predictions, 10)

[1] No No No No No No No No No No
Levels: No Yes

### Q7: Get Probability Predictions

Get class probability predictions on the holdout set. Store the result in **`probabilities`**.

**Expected output:** Print the first 10 rows — a data frame with two columns (`No` and `Yes`) showing the probability of each class.

In [15]:
probabilities <- predict(model, holdout.df, type="prob")
head(probabilities, 10)

,No,Yes
,<dbl>,<dbl>
1,1.0,0.0
2,1.0,0.0
3,0.8,0.2
4,1.0,0.0
5,1.0,0.0
6,1.0,0.0
7,0.8,0.2
8,1.0,0.0
9,1.0,0.0


### Q8: Tune k Using Cross-Validation

Tune the k-NN model using **5-fold cross-validation** to find the best k.

- Store the train control object in **`trControl`**
- Test `k = 1, 3, 5, 7, 9, 11, 13`
- Store the tuned model in **`model_tuned`**

**Expected output:** A table showing accuracy for each tested k value, with the best k highlighted.

In [33]:

trControl <- trainControl(method="cv", number=5, allowParallel=TRUE)
model_tuned <- train(Default ~ ., data=train.df,
method="knn", preProcess=c("center", "scale"),
tuneGrid=expand.grid(k=c(1, 3, 5, 7, 9, 11, 13)),
trControl=trControl) 
model_tuned


k-Nearest Neighbors 

153208 samples
     7 predictor
     2 classes: 'No', 'Yes' 

Pre-processing: centered (7), scaled (7) 
Resampling: Cross-Validated (5 fold) 
Summary of sample sizes: 122567, 122567, 122565, 122567, 122566 
Resampling results across tuning parameters:

  k   Accuracy   Kappa     
   1  0.8150097  0.09064282
   3  0.8584473  0.09496264
   5  0.8725067  0.08944736
   7  0.8778654  0.08420712
   9  0.8802478  0.07205296
  11  0.8821014  0.06977603
  13  0.8828586  0.06253057

Accuracy was used to select the optimal model using the largest value.
The final value used for the model was k = 13.

### Q9: Train Final Model with Best k

Train a final model on **all of `knn.df`** using the best k from Q8. Store it in **`final_model`**.

**Expected output:** Model summary trained on the full dataset with the optimal k value.

In [17]:
# Your code here

---

# Part 2: Factor Analysis

**Dataset:** `Factor-Analysis-Example.csv`

This dataset contains survey responses where respondents rated an instructor on 9 attributes (1–10 scale):
**Expect**, **Entertain**, **Comm**, **Expert**, **Motivate**, **Caring**, **Charisma**, **Passion**, **Friendly**

Your goal is to discover the hidden factors underlying these ratings.

### Q10: Load Libraries

Load the following libraries:
- `psych`
- `corrplot`

In [18]:
# Your code here

### Q11: Load and Explore the Data

Load the factor analysis dataset and store it in **`fa_data`**.

**Note:** This CSV may have extra empty columns. If so, keep only the first 9 columns (Expect through Friendly).

**Expected output:** `str()` should show 9 numeric variables and approximately 125 observations.

In [19]:
# Your code here

### Q12: Correlation Matrix and Heatmap

Compute the correlation matrix of `fa_data` and store it in **`cor_matrix`**. Then visualize it as a heatmap.

**Expected output:** A heatmap showing pairwise correlations between all 9 variables with correlation coefficients displayed. Look for blocks of variables that correlate strongly with each other.

In [20]:
# Your code here

### Q13: KMO and Bartlett's Tests

Run the KMO test and Bartlett's test of sphericity to verify that the data is suitable for factor analysis.

- Store the KMO result in **`kmo_result`**
- Store the Bartlett's result in **`bartlett_result`**

**Expected output:** Print the overall KMO MSA value (check if ≥ 0.60), and the Bartlett's chi-square value and p-value (check if p < 0.05). Both must pass to proceed.

In [21]:
# Your code here

### Q14: Parallel Analysis

Run a parallel analysis on `fa_data` to determine the number of factors to extract.

**Expected output:** A scree plot and a text message suggesting the number of factors. Record this number — you will use it in Q15.

In [22]:
# Your code here

### Q15: Extract Factors with Varimax Rotation

Extract factors from `fa_data` using **varimax rotation** with the number of factors suggested by Q14. Store the result in **`fa_result`**.

Print the loadings with a cutoff of **0.3** (sorted).

**Expected output:** A factor loadings table showing which variables load on which factors. Values below 0.3 are hidden for clarity.

In [23]:
# Your code here

### Q16: Factor Diagram

Create a visual diagram of the factor structure from `fa_result`.

**Expected output:** A diagram with ovals (factors) connected by arrows to rectangles (variables). Thicker arrows indicate stronger factor loadings.

In [24]:
# Your code here

### Q17: Cronbach's Alpha

Based on the factor loadings from Q15, identify which variables belong to each factor. Pick **one** of your factors and compute Cronbach's Alpha for those variables.

- Store the result in **`alpha_result`**

**Expected output:** Print the raw alpha value. Check if it is ≥ 0.70 (acceptable reliability).

In [25]:
# Your code here

---

# Part 3: Conjoint Analysis

**Dataset:** `pizza_data.csv`

A pizza company wants to understand which product features drive customer preference. Respondents ranked 16 different pizza profiles (1 = most preferred, 16 = least preferred).

**Attributes:** `crust`, `cheese`, `size`, `toppings`, `spicy`

**Preference variable:** `ranking`

### Q18: Load the Data

Load the pizza conjoint dataset and store it in **`pizza.df`**.

**Expected output:** A data frame with 16 rows and 9 columns showing all pizza profiles and their rankings.

In [26]:
# Your code here

### Q19: Regression-Based Conjoint Analysis

Fit a linear regression predicting `ranking` from `crust`, `cheese`, `size`, `toppings`, and `spicy`. Store the model in **`pizza_model`**.

**Expected output:** A regression summary where each coefficient is a **part-worth utility** showing how much each feature level changes the ranking relative to the reference level. Check which coefficients are significant (p < 0.05).

In [27]:
# Your code here

### Q20: Visualize Mean Ranking by Attribute

Create bar plots showing the **mean ranking** for each level of at least **three** of the five attributes (`crust`, `cheese`, `size`, `toppings`, `spicy`).

**Expected output:** Bar charts where the height of each bar is the average ranking for that attribute level. **Remember:** lower ranking = more preferred, so shorter bars indicate higher preference.

In [28]:
# Your code here